### 2 · Ingestion — Embed Processed Chunks into FAISS

This notebook loads the enriched chunks from `data/processed/chunks_enriched.json`,
embeds them using OpenAI's `text-embedding-3-small`, and persists a FAISS index
to `data/vectorstore/`.

### Pipeline
```
data/processed/chunks_enriched.json
  → Load JSON into LangChain Documents
  → Embed with OpenAI text-embedding-3-small
  → Build FAISS index
  → Persist to data/vectorstore/
  → Validate with sample queries
```

In [1]:
import json
from pathlib import Path
from pprint import pprint

from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from dotenv import load_dotenv

load_dotenv(override=True)

PROCESSED_DIR = Path("../data/processed")
VECTORSTORE_DIR = Path("../data/vectorstore")
VECTORSTORE_DIR.mkdir(parents=True, exist_ok=True)

EMBEDDING_MODEL = "text-embedding-3-small"

print(f"Processed dir: {PROCESSED_DIR.resolve()}")
print(f"Vectorstore dir: {VECTORSTORE_DIR.resolve()}")

Processed dir: /Users/sauravmajumdar/Developer/AI/move-mind-ai/data/processed
Vectorstore dir: /Users/sauravmajumdar/Developer/AI/move-mind-ai/data/vectorstore


### Step 1 — Load Processed Chunks

Reconstruct `Document` objects from the JSON exported by notebook 1.

In [ ]:
chunks_path = PROCESSED_DIR / "chunks_enriched.json"

with open(chunks_path, "r", encoding="utf-8") as f:
    raw_chunks = json.load(f)

documents = [
    Document(page_content=c["page_content"], metadata=c["metadata"])
    for c in raw_chunks
]

print(f"Loaded {len(documents)} documents from {chunks_path.name}")
print(f"\nSample document:")
print(f"  Content: {documents[0].page_content[:150]}...")
print(f"  Metadata keys: {list(documents[0].metadata.keys())}")

Loaded 292 documents from chunks_enriched.json

Sample document:
  Content: # Code Quality & Style Guide for Developers  
> **Parent:** [Architecture](./architecture.md)
> **Last Updated:** March 2026  
---...
  Metadata keys: ['source', 'doc_title', 'doc_type', 'audience', 'owner', 'status', 'version', 'last_modified', 'section', 'heading_hierarchy', 'chunk_index']


### Step 2 — Create Embeddings & Build FAISS Index

This step calls the OpenAI embeddings API for all chunks and builds an in-memory
FAISS index. Cost estimate for `text-embedding-3-small`:
- ~292 chunks × ~500 chars avg ≈ ~36K tokens ≈ **$0.0007**

In [3]:
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

print(f"Embedding {len(documents)} chunks with '{EMBEDDING_MODEL}'...")
vectorstore = FAISS.from_documents(documents, embeddings)
print(f"FAISS index built — {vectorstore.index.ntotal} vectors, dimension {vectorstore.index.d}")

Embedding 292 chunks with 'text-embedding-3-small'...
FAISS index built — 292 vectors, dimension 1536


### Step 3 — Persist to Disk

Save the FAISS index + docstore to `data/vectorstore/` so the app can load it
via `FAISS.load_local()` without re-embedding.

In [4]:
vectorstore.save_local(str(VECTORSTORE_DIR))

# Verify saved files
saved_files = list(VECTORSTORE_DIR.iterdir())
print("Saved files:")
for f in saved_files:
    print(f"  {f.name:<30} {f.stat().st_size / 1024:.1f} KB")

Saved files:
  index.faiss                    1752.0 KB
  index.pkl                      252.7 KB
  .gitkeep                       0.0 KB


### Step 4 — Validate: Load from Disk & Test Retrieval

Reload the persisted index (exactly how `app/rag/retriever.py` will use it)
and run sample similarity searches to verify everything works.

In [5]:
# Reload from disk (same as app/rag/retriever.py)
loaded_store = FAISS.load_local(
    str(VECTORSTORE_DIR),
    embeddings,
    allow_dangerous_deserialization=True,
)
print(f"Reloaded index: {loaded_store.index.ntotal} vectors")

retriever = loaded_store.as_retriever(search_kwargs={"k": 5})

Reloaded index: 292 vectors


In [6]:
# ── Test query 1: Architecture question ──────────────────────────────────────

query = "How is state management handled in the application?"
results = retriever.invoke(query)

print(f"Query: {query}")
print(f"Results: {len(results)} chunks\n")
for i, doc in enumerate(results):
    print(f"  [{i}] {doc.metadata['source']:<45} section: {doc.metadata['section'][:50]}")
    print(f"      {doc.page_content[:120]}...\n")

Query: How is state management handled in the application?
Results: 5 chunks

  [0] architecture/overview.md                      section: When to Use What
      ## State Management Strategy  
### When to Use What  
| State Type | Technology | Examples |
|---|---|---|
| **Server da...

  [1] architecture/code-quality.md                  section: What Goes Where
      ## State Management Rules  
### What Goes Where  
| State Type | Technology | Example |
|---|---|---|
| Server data (jou...

  [2] architecture/redux.md                         section: Overview
      ## Overview  
The AMS Admin Tool uses **Redux Toolkit** for global client-side state management. Redux handles state tha...

  [3] features/journey-canvas.md                    section: Builder Slice (`builderSlice`)
      ## State Management  
### Builder Slice (`builderSlice`)  
Used by Journey Builder for full CRUD operations:  
| Action ...

  [4] architecture/overview.md                      section: Overview
      ## Ov

In [7]:
# ── Test query 2: Feature question ───────────────────────────────────────────

query = "What is the journey canvas feature?"
results = retriever.invoke(query)

print(f"Query: {query}")
print(f"Results: {len(results)} chunks\n")
for i, doc in enumerate(results):
    print(f"  [{i}] {doc.metadata['source']:<45} section: {doc.metadata['section'][:50]}")
    print(f"      {doc.page_content[:120]}...\n")

Query: What is the journey canvas feature?
Results: 5 chunks

  [0] features/journey-canvas.md                    section: Overview
      ## Overview  
The Journey Canvas is the visual graph editor at the heart of the AMS Admin Tool. It renders CMS2 journey ...

  [1] features/journey-canvas.md                    section: Journey Canvas — Technical Documentation
      # Journey Canvas — Technical Documentation  
> **Version:** 1.0
> **Last Updated:** March 2026
> **Owner:** Admin Tool E...

  [2] index.md                                      section: Features
      ## Features  
Deep-dive documentation for major product features.  
| Document | Description |
|---|---|
| [Journey Buil...

  [3] features/journey-canvas.md                    section: Known Limitations
      ## Known Limitations  
- **No undo/redo** — canvas operations (delete node, connect, disconnect) have no undo mechanism ...

  [4] architecture/overview.md                      section: Glossary
      | **Journey Refer

In [8]:
# ── Test query 3: Metadata filtering ─────────────────────────────────────────

query = "code quality"
results_all = loaded_store.similarity_search(query, k=5)
results_arch = loaded_store.similarity_search(
    query, k=5, filter={"doc_type": "architecture"}
)

print("Without filter:")
for doc in results_all:
    print(f"  {doc.metadata['doc_type']:<15} {doc.metadata['source']}")

print(f"\nWith filter (doc_type='architecture'):")
for doc in results_arch:
    print(f"  {doc.metadata['doc_type']:<15} {doc.metadata['source']}")

Without filter:
  architecture    architecture/code-quality.md
  operations      operations/code-quality-audit-plan.md
  architecture    architecture/code-quality.md
  operations      operations/code-quality-audit-plan.md
  feature         features/action-builder.md

With filter (doc_type='architecture'):
  architecture    architecture/code-quality.md
  architecture    architecture/code-quality.md
  architecture    architecture/code-quality.md
  architecture    architecture/code-quality.md
  architecture    architecture/code-quality.md


### Summary

| Step | What | Output |
|------|------|--------|
| 1 | Load processed JSON | `Document` objects with enriched metadata |
| 2 | Embed with OpenAI | FAISS in-memory index |
| 3 | Persist to disk | `data/vectorstore/index.faiss` + `index.pkl` |
| 4 | Validate | Similarity search + metadata filtering |

The vectorstore is now ready for `app/rag/retriever.py` to load via:
```python
FAISS.load_local(settings.VECTORSTORE_PATH, embeddings, allow_dangerous_deserialization=True)
```